# Phase 2 - 4: Model Experiments & Noise Robustness
This notebook covers:
1. Baseline CNN Classifier architecture and predictions
2. Convolutional Denoising Autoencoder (DAE) latent representations
3. Controlled noise experiments across varying sigma levels
4. DAE Reconstruction metrics (PSNR, MSE)
5. Optional Extension: Pretrained Encoder feature extractor comparison

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

from src import config
from src.models import build_cnn_classifier, build_denoising_autoencoder, build_encoder_classifier
from src.prepare_data import load_dataset_arrays, add_gaussian_noise
from src.train_denoising_ae import calculate_psnr

print("TensorFlow:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

## 1. Inspect Model Architectures

In [ ]:
cnn_model = build_cnn_classifier()
print("=== CNN Classifier Summary ===")
cnn_model.summary()

dae_model, encoder_model = build_denoising_autoencoder()
print("=== Denoising Autoencoder Summary ===")
dae_model.summary()

## 2. Noise Degradation Sweep Experiment
Evaluating how different levels of Gaussian noise affect image quality.

In [ ]:
X_test_clean, y_test = load_dataset_arrays("test")
sample_clean = X_test_clean[0]

noise_levels = [0.05, 0.15, 0.25, 0.40, 0.60]
fig, axes = plt.subplots(1, len(noise_levels) + 1, figsize=(16, 3))

axes[0].imshow(sample_clean)
axes[0].set_title("Original Clean", fontweight="bold")
axes[0].axis("off")

for idx, sigma in enumerate(noise_levels):
    noisy = add_gaussian_noise(np.expand_dims(sample_clean, 0), noise_factor=sigma)[0]
    psnr_val = calculate_psnr(sample_clean, noisy)
    axes[idx + 1].imshow(noisy)
    axes[idx + 1].set_title(f"σ = {sigma}\nPSNR: {psnr_val:.1f} dB")
    axes[idx + 1].axis("off")

plt.suptitle("Simulated Visual Sensor Noise Degradation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Optional Extension: Pretrained Encoder Feature Extractor
As outlined in PRD Section 8.3, we can reuse the DAE encoder as a feature extractor for classification.

In [ ]:
pretrained_classifier = build_encoder_classifier(encoder_model, freeze_encoder=True)
pretrained_classifier.summary()